In [1]:
%%writefile ..\notebooks\unittest_project.py

import pandas as pd
import pytest

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# Test data paths
# ---------------------------------------------------------

TRAIN_URL = (
    "https://huggingface.co/datasets/"
    "ourafla/Mental-Health_Text-Classification_Dataset/"
    "resolve/main/mental_heath_unbanlanced.csv"
)

TEST_URL = (
    "https://huggingface.co/datasets/"
    "ourafla/Mental-Health_Text-Classification_Dataset/"
    "resolve/main/mental_health_combined_test.csv"
)


# ---------------------------------------------------------
# Expected classes
# ---------------------------------------------------------

EXPECTED_CLASSES = {
    "Anxiety",
    "Depression",
    "Normal",
    "Suicidal"
}


# ---------------------------------------------------------
# Fixtures
# ---------------------------------------------------------

@pytest.fixture(scope="module")
def train_data():
    """Load the training dataset."""
    return pd.read_csv(TRAIN_URL)


@pytest.fixture(scope="module")
def test_data():
    """Load the held-out test dataset."""
    return pd.read_csv(TEST_URL)


# ---------------------------------------------------------
# Data validation tests
# ---------------------------------------------------------

def test_training_data_columns(train_data):
    """Training data must contain text and status columns."""
    assert "text" in train_data.columns
    assert "status" in train_data.columns


def test_test_data_columns(test_data):
    """Test data must contain text and status columns."""
    assert "text" in test_data.columns
    assert "status" in test_data.columns


def test_training_data_not_empty(train_data):
    """Training dataset must contain records."""
    assert len(train_data) > 0


def test_test_data_not_empty(test_data):
    """Test dataset must contain records."""
    assert len(test_data) > 0


def test_no_missing_values(train_data, test_data):
    """Text and labels must not contain missing values."""
    assert train_data["text"].notna().all()
    assert train_data["status"].notna().all()

    assert test_data["text"].notna().all()
    assert test_data["status"].notna().all()


def test_expected_classes(train_data, test_data):
    """Both datasets must contain the four expected classes."""

    train_classes = set(train_data["status"].unique())
    test_classes = set(test_data["status"].unique())

    assert train_classes == EXPECTED_CLASSES
    assert test_classes == EXPECTED_CLASSES


# ---------------------------------------------------------
# Model pipeline tests
# ---------------------------------------------------------

def test_tfidf_creates_features(train_data):
    """TF-IDF should successfully convert text into numerical features."""

    vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )

    X = vectorizer.fit_transform(train_data["text"])

    assert X.shape[0] == len(train_data)
    assert X.shape[1] > 0


def test_model_can_train_and_predict(train_data, test_data):
    """
    The final word-level TF-IDF + Logistic Regression pipeline
    should train successfully and produce predictions.
    """

    vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(train_data["text"])
    X_test = vectorizer.transform(test_data["text"])

    model = LogisticRegression(
        C=2.0,
        max_iter=2000
    )

    model.fit(X_train, train_data["status"])

    predictions = model.predict(X_test)

    assert len(predictions) == len(test_data)


def test_predictions_are_valid_classes(train_data, test_data):
    """Model predictions must belong to the four expected classes."""

    vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(train_data["text"])
    X_test = vectorizer.transform(test_data["text"])

    model = LogisticRegression(
        C=2.0,
        max_iter=2000
    )

    model.fit(X_train, train_data["status"])

    predictions = model.predict(X_test)

    assert set(predictions).issubset(EXPECTED_CLASSES)

Writing ..\notebooks\unittest_project.py


In [2]:
from scipy.sparse import hstack

In [3]:
def test_final_model_can_train_and_predict(train_data, test_data):
    """Final word + character TF-IDF model should train and predict."""

    # Word-level TF-IDF
    word_tfidf = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )

    X_train_word = word_tfidf.fit_transform(train_data["text"])
    X_test_word = word_tfidf.transform(test_data["text"])

    # Character-level TF-IDF
    char_tfidf = TfidfVectorizer(
        analyzer="char",
        max_features=30000,
        ngram_range=(3, 5),
        sublinear_tf=True
    )

    X_train_char = char_tfidf.fit_transform(train_data["text"])
    X_test_char = char_tfidf.transform(test_data["text"])

    # Combine word + character features
    X_train_combined = hstack([
        X_train_word,
        X_train_char
    ])

    X_test_combined = hstack([
        X_test_word,
        X_test_char
    ])

    # Train final model
    model = LogisticRegression(
        C=2.0,
        max_iter=2000
    )

    model.fit(
        X_train_combined,
        train_data["status"]
    )

    predictions = model.predict(X_test_combined)

    # Checks
    assert len(predictions) == len(test_data)
    assert set(predictions).issubset(EXPECTED_CLASSES)

In [4]:
!python -m pytest ..\notebooks\unittest_project.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\bhatn\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\bhatn\Data703-PythonProject\mental_health_text_classification\notebooks
plugins: anyio-4.10.0
collecting ... collected 9 items

unittest_project.py::test_training_data_columns PASSED                   [ 11%]
unittest_project.py::test_test_data_columns PASSED                       [ 22%]
unittest_project.py::test_training_data_not_empty PASSED                 [ 33%]
unittest_project.py::test_test_data_not_empty PASSED                     [ 44%]
unittest_project.py::test_no_missing_values PASSED                       [ 55%]
unittest_project.py::test_expected_classes PASSED                        [ 66%]
unittest_project.py::test_tfidf_creates_features PASSED                  [ 77%]
unittest_project.py::test_model_can_train_and_predict PASSED             [ 88%]
unittest_pr